# Generation of ketamine data pair

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals
from Functions.time_frequency import spectrogram

## Inspect how to generate synthetic data pairs

In [ ]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2]
omega_list = [2*np.pi*1, 2*np.pi*10]
sigma_list = [3, 2]
factor_list = [1, 1]

# --- Generate EEG data
t, y = get_mixed_OU_signals(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[2].set_title('PSD')
axes[1].sharex(axes[0])

plt.show()

__Take part of signal to enhance__

In [18]:
mask_f = f_spectro >= 20
f_M = f_spectro[mask_f]
M = spectro[mask_f, :].copy()

# --- Display
fig, axes = plt.subplots(2,  constrained_layout = True)
axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')
axes[0].set_title('Spectrogram')
axes[1].plot(f_M, np.log2(np.median(M, axis = 1)))
axes[1].set_title('PSD')

plt.show()

__Example of Watershed applied to three blobs__

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import label, regionprops

# 1. Create a dummy spectrogram with 3 overlapping "hills"
x, y = np.mgrid[-5:5:0.05, -5:5:0.05]
hill1 = np.exp(-((x-1)**2 + (y-1)**2))
hill2 = np.exp(-((x+1)**2 + (y+1)**2)) * 0.8
hill3 = np.exp(-((x-1.5)**2 + (y+1.5)**2)) * 0.9
spectrogram = hill1 + hill2 + hill3

# 2. Pre-process: Smooth slightly to handle noise
smoothed = gaussian_filter(spectrogram, sigma=1.0)

# 3. Find the local maxima (the peaks of the hills)
coordinates = peak_local_max(smoothed, min_distance=10, threshold_abs=0.1)

# Create a boolean mask of the peaks for the watershed markers
markers_mask = np.zeros(smoothed.shape, dtype=bool)
markers_mask[tuple(coordinates.T)] = True

# Fixed: Use return_num=True to safely unpack two values in skimage
markers, num_features = label(markers_mask, return_num=True)

# 4. Run Watershed Segmentation
labels = watershed(-smoothed, markers, mask=smoothed > 0.05)

# 5. Extract information about your hills
print(f"Found {num_features} distinct hills.")
for prop in regionprops(labels, intensity_image=spectrogram):
    print(f"Hill {prop.label}:")
    print(f"  - Peak Intensity: {prop.max_intensity:.2f}")
    print(f"  - Center (Barycenter): {prop.centroid}")
    print(f"  - Area (pixels): {prop.area}")

# 6. Plot the results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(spectrogram, origin='lower', cmap='viridis')
axes[0].set_title("Original Spectrogram")

axes[1].imshow(smoothed, origin='lower', cmap='viridis')
axes[1].autoscale(False)
# Fixed: standard peak_local_max returns (row, col), which maps to (y, x) in plot
axes[1].plot(coordinates[:, 1], coordinates[:, 0], 'r.', markersize=10)
axes[1].set_title("Detected Local Maxima (Peaks)")

axes[2].imshow(labels, origin='lower', cmap='nipy_spectral')
axes[2].set_title("Watershed Segmentation (Hills)")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Found 3 distinct hills.
Hill 1:
  - Peak Intensity: 0.80
  - Center (Barycenter): (np.float64(77.3632958801498), np.float64(78.83021223470662))
  - Area (pixels): 3204.0
Hill 2:
  - Peak Intensity: 1.00
  - Center (Barycenter): (np.float64(120.89434608794974), np.float64(122.42518560822387))
  - Area (pixels): 3502.0
Hill 3:
  - Peak Intensity: 0.90
  - Center (Barycenter): (np.float64(132.62520404831864), np.float64(66.96800522363695))
  - Area (pixels): 3063.0


C:\Users\PO\AppData\Local\Temp\ipykernel_29244\62817629.py:35: FutureWarning: `RegionProperties.max_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_max` instead. 
  print(f"  - Peak Intensity: {prop.max_intensity:.2f}")


__Watershed applied to M matrix__

In [13]:
fig, axis = plt.subplots(1)
axis.hist(np.ravel(M))
plt.show()

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import label, regionprops

# =====================================================================
# STEP 2: DYNAMIC RANGE NORMALIZATION [CRITICAL FOR SMALL VALUES]
# =====================================================================
# We map the data dynamically to a 0.0 to 1.0 range. 
# This prevents hardcoded thresholds from breaking if magnitudes shift.
data_min = M.min()
data_max = M.max()
M_norm = (M - data_min) / (data_max - data_min)

# =====================================================================
# STEP 3: PRE-PROCESSING (SMOOTHING)
# =====================================================================
# Gaussian blur suppresses pixel noise so it isn't misidentified as tiny peaks.
# Increase sigma if you get 'fake' micro-peaks; decrease it if hills blur together.
smoothed = gaussian_filter(M_norm, sigma=1.0)

# =====================================================================
# STEP 4: IDENTIFY LOCAL PEAKS (WATERSHED MARKERS)
# =====================================================================
# min_distance: Minimum pixel distance allowed between distinct peaks.
# threshold_abs: Minimum height on our [0,1] scale to consider a peak.
#                0.10 means a hill must be at least 10% higher than the global minimum.
coordinates = peak_local_max(smoothed, min_distance=5, threshold_abs=0.05)

# Build a boolean mask array matching the image shape, setting peak locations to True
markers_mask = np.zeros(smoothed.shape, dtype=bool)
markers_mask[tuple(coordinates.T)] = True

# Convert unique peaks into unique integer IDs (1, 2, 3...) to act as seeds.
# return_num=True ensures cross-compatibility with various scikit-image versions.
markers, num_features = label(markers_mask, return_num=True)

# =====================================================================
# STEP 5: RUN WATERSHED SEGMENTATION
# =====================================================================
# 1. We pass '-smoothed' (negative) because watershed looks for basins/valleys.
#    Inverting makes our high peaks the lowest points.
# 2. 'mask=smoothed > 0.05' acts as a global floor threshold. Any pixel below 
#    5% of our normalized range is ignored completely as background noise.
labels = watershed(-smoothed, markers, mask=smoothed > 0.05)

# =====================================================================
# STEP 6: EXTRACT PROPERTIES (USING ORIGINAL VALUES)
# =====================================================================
print(f"=== Detection Summary ===")
print(f"Found {num_features} distinct hills.\n")

# CRITICAL: We pass the ORIGINAL un-normalized 'spectrogram_raw' here 
# so the printed measurements reflect your actual physical units.
for prop in regionprops(labels, intensity_image=M):
    print(f"Hill ID {prop.label}:")
    print(f"  - Peak Value (Max Intensity): {prop.max_intensity:.4f}")
    print(f"  - Center Coordinates (Y, X):  ({prop.centroid[0]:.1f}, {prop.centroid[1]:.1f})")
    print(f"  - Total Footprint Area:       {prop.area} pixels")
    print("-" * 35)

# =====================================================================
# STEP 7: VISUALIZATION
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Raw input data
im0 = axes[0].imshow(M, origin='lower', cmap='viridis')
axes[0].set_title("1. Original Raw Spectrogram")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Plot 2: Peak detection step
im1 = axes[1].imshow(smoothed, origin='lower', cmap='viridis')
axes[1].autoscale(False)
# Coordinates are extracted as (row, col) which correlates to (Y, X) on a plot
axes[1].plot(coordinates[:, 1], coordinates[:, 0], 'r.', markersize=12, label='Found Peaks')
axes[1].set_title("2. Smoothed Data + Maxima")
axes[1].legend(loc='upper right')

# Plot 3: Resulting separated hills
im2 = axes[2].imshow(labels, origin='lower', cmap='nipy_spectral')
axes[2].set_title("3. Final Watershed Clusters")

# Clean up layout display
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

=== Detection Summary ===
Found 40 distinct hills.

Hill ID 1:
  - Peak Value (Max Intensity): 0.0191
  - Center Coordinates (Y, X):  (7.5, 39.7)
  - Total Footprint Area:       185.0 pixels
-----------------------------------
Hill ID 2:
  - Peak Value (Max Intensity): 0.0076
  - Center Coordinates (Y, X):  (6.4, 67.3)
  - Total Footprint Area:       277.0 pixels
-----------------------------------
Hill ID 3:
  - Peak Value (Max Intensity): 0.0130
  - Center Coordinates (Y, X):  (6.2, 135.7)
  - Total Footprint Area:       512.0 pixels
-----------------------------------
Hill ID 4:
  - Peak Value (Max Intensity): 0.0053
  - Center Coordinates (Y, X):  (10.3, 122.9)
  - Total Footprint Area:       74.0 pixels
-----------------------------------
Hill ID 5:
  - Peak Value (Max Intensity): 0.0038
  - Center Coordinates (Y, X):  (14.0, 95.1)
  - Total Footprint Area:       53.0 pixels
-----------------------------------
Hill ID 6:
  - Peak Value (Max Intensity): 0.0053
  - Center Coordinate

C:\Users\PO\AppData\Local\Temp\ipykernel_29244\860770619.py:59: FutureWarning: `RegionProperties.max_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_max` instead. 
  print(f"  - Peak Value (Max Intensity): {prop.max_intensity:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import label, regionprops

# (Assuming M is your 2D numpy array loaded here)

# =====================================================================
# STEP 2: DYNAMIC RANGE NORMALIZATION
# =====================================================================
data_min = M.min()
data_max = M.max()
M_norm = (M - data_min) / (data_max - data_min)

# =====================================================================
# STEP 3: PRE-PROCESSING (SMOOTHING)
# =====================================================================
smoothed = gaussian_filter(M_norm, sigma=1.0)

# =====================================================================
# STEP 4: FIND VALLEYS (LOCAL MINIMA) AS MARKERS
# =====================================================================
# To find valleys (minima), we look for peaks on the INVERTED image (1.0 - smoothed).
# This is usually much easier to tune because backgrounds or wide valleys are stable.
inverted_smoothed = 1.0 - smoothed

# Adjust 'min_distance' based on how far apart your valleys generally are.
# 'threshold_abs=0.50' means we only look for valleys that drop significantly 
# below the hill tops (at least 50% down from the max peak).
valley_coordinates = peak_local_max(inverted_smoothed, min_distance=10, threshold_abs=0.20)

# Build the markers array from the detected valleys
markers_mask = np.zeros(smoothed.shape, dtype=bool)
markers_mask[tuple(valley_coordinates.T)] = True
markers, num_features = label(markers_mask, return_num=True)

# =====================================================================
# STEP 5: RUN WATERSHED SEGMENTATION (FLOODING THE VALLEYS)
# =====================================================================
# 1. We pass 'smoothed' directly (NOT negative). The algorithm starts at 
#    our valley seeds and crawls UP toward the hill tops.
# 2. We invert the mask: 'mask=smoothed < 0.95'. This tells the algorithm to
#    ignore the absolute very tip-tops of your hills, forcing it to draw a 
#    clean perimeter boundary *around* each hill peak.
labels = watershed(smoothed, markers, mask=smoothed < 0.95)

# =====================================================================
# STEP 6: EXTRACT PROPERTIES (USING ORIGINAL VALUES)
# =====================================================================
print(f"=== Valley-Based Detection Summary ===")
print(f"Found {num_features} distinct background valley zones.\n")

for prop in regionprops(labels, intensity_image=M):
    print(f"Hill Enclosure ID {prop.label}:")
    print(f"  - Peak Value inside this zone: {prop.max_intensity:.4f}")
    print(f"  - Center Coordinates (Y, X):  ({prop.centroid[0]:.1f}, {prop.centroid[1]:.1f})")
    print(f"  - Enclosed Area:              {prop.area} pixels")
    print("-" * 35)

# =====================================================================
# STEP 7: VISUALIZATION
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Raw input data
im0 = axes[0].imshow(M, origin='lower', cmap='viridis')
axes[0].set_title("1. Original Raw Spectrogram")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Plot 2: Valley detection step
im1 = axes[1].imshow(smoothed, origin='lower', cmap='viridis')
axes[1].autoscale(False)
# Mark the detected valleys in blue
axes[1].plot(valley_coordinates[:, 1], valley_coordinates[:, 0], 'b.', markersize=12, label='Found Valleys')
axes[1].set_title("2. Smoothed Data + Valleys (Seeds)")
axes[1].legend(loc='upper right')

# Plot 3: Resulting separated hill footprints
im2 = axes[2].imshow(labels, origin='lower', cmap='nipy_spectral')
axes[2].set_title("3. Watershed Hill Enclosures")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

=== Valley-Based Detection Summary ===
Found 11 distinct background valley zones.

Hill Enclosure ID 1:
  - Peak Value inside this zone: 0.0155
  - Center Coordinates (Y, X):  (7.7, 178.6)
  - Enclosed Area:              774.0 pixels
-----------------------------------
Hill Enclosure ID 2:
  - Peak Value inside this zone: 0.0191
  - Center Coordinates (Y, X):  (10.2, 27.2)
  - Enclosed Area:              447.0 pixels
-----------------------------------
Hill Enclosure ID 3:
  - Peak Value inside this zone: 0.0045
  - Center Coordinates (Y, X):  (15.0, 54.6)
  - Enclosed Area:              163.0 pixels
-----------------------------------
Hill Enclosure ID 4:
  - Peak Value inside this zone: 0.0082
  - Center Coordinates (Y, X):  (22.9, 43.7)
  - Enclosed Area:              421.0 pixels
-----------------------------------
Hill Enclosure ID 5:
  - Peak Value inside this zone: 0.0033
  - Center Coordinates (Y, X):  (25.9, 133.4)
  - Enclosed Area:              222.0 pixels
-----------------

C:\Users\PO\AppData\Local\Temp\ipykernel_29244\2439921375.py:57: FutureWarning: `RegionProperties.max_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_max` instead. 
  print(f"  - Peak Value inside this zone: {prop.max_intensity:.4f}")
